In [ ]:
"""
Objetivo: Executar os modelos do Rolling-Window VAR. O script gera os outputs 
da Subseção 4.3 (foco na janela baseline de 48 meses e suas âncoras de crise) 
e os painéis de robustez multijanela (36 a 84 meses) para o Apêndice A.2.
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})

# ==========================================
# 0. PARÂMETROS GERAIS
# ==========================================
ARQUIVO_CSV = '07_base_rolling_var_fred.csv'
ORDEM_CHOLESKY = ['TED_Spread', 'Desemprego', 'Fed_Funds'] 
LISTA_JANELAS = [36, 48, 60, 72, 84]
JANELA_BASELINE = 48
LAG_VAR = 1
HORIZONTE_IRF = 12       
HORIZONTE_FEVD = 6       
HORIZONTE_PREVISAO = 3   
N_REPLICACOES = 500      
SIGNIF = 0.10            
SEMENTE = 42

REGIMES_IRF = {
    'Pré-Crise (2006)': '2006-12-01',
    'Subprime (2007)': '2007-12-01',
    'Lehman Bros. (2008)': '2008-12-01',
}
CORES_REGIMES = {
    'Pré-Crise (2006)': '#1f77b4',
    'Subprime (2007)': '#ff7f0e',
    'Lehman Bros. (2008)': '#d62728',
}

# ==========================================
# 1. PREPARAÇÃO DO VETOR MACROECONÔMICO
# ==========================================
if not os.path.exists(ARQUIVO_CSV):
    raise SystemExit(f"[ERRO] Arquivo '{ARQUIVO_CSV}' não encontrado. Execute a etapa anterior.")

df = pd.read_csv(ARQUIVO_CSV, parse_dates=['Data'], index_col='Data')[ORDEM_CHOLESKY]
df = df.diff().dropna() 

idx_ted = ORDEM_CHOLESKY.index('TED_Spread')
idx_un = ORDEM_CHOLESKY.index('Desemprego')
idx_fed = ORDEM_CHOLESKY.index('Fed_Funds')

# ==========================================
# 2. OUTPUTS DA SUBSEÇÃO 4.3 (Janela Padrão de 48 Meses)
# ==========================================
def gerar_outputs_secao_43():
    print("[*] Gerando gráficos da Subseção 4.3 (Janela baseline de 48 meses)...")
    
    # A. Painel de IRF por Regimes (Figura 7 equivalente)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    meses = np.arange(HORIZONTE_IRF + 1)
    
    for idx, (nome, data_fim_str) in enumerate(REGIMES_IRF.items()):
        data_fim = pd.to_datetime(data_fim_str)
        data_inicio = data_fim - pd.DateOffset(months=JANELA_BASELINE)
        janela = df.loc[data_inicio:data_fim]
        
        res = VAR(janela).fit(LAG_VAR)
        irf = res.irf(HORIZONTE_IRF)
        resposta = irf.orth_irfs[:, idx_fed, idx_ted]
        inf, sup = irf.errband_mc(orth=True, repl=N_REPLICACOES, signif=SIGNIF, seed=SEMENTE)
        
        ax = axes[idx]
        ax.plot(meses, resposta, color=CORES_REGIMES[nome], linewidth=2.5)
        ax.fill_between(meses, inf[:, idx_fed, idx_ted], sup[:, idx_fed, idx_ted], color=CORES_REGIMES[nome], alpha=0.15)
        ax.axhline(0, color='black', linewidth=1, linestyle='--')
        ax.set_title(nome, fontsize=11, fontweight='bold')
        ax.set_xlabel('Meses após o choque', fontsize=10, fontweight='bold')
        if idx == 0:
            ax.set_ylabel('Resposta da Fed (Variação)', fontsize=10, fontweight='bold')
            
    plt.tight_layout()
    plt.savefig('09b_painel_irf_regimes_corrigido.png', dpi=300, bbox_inches='tight')
    plt.close()

    # B. Rolling-FEVD para a janela de 48 meses (Figura 8 equivalente)
    datas, linhas = [], []
    for i in range(JANELA_BASELINE, len(df)):
        janela = df.iloc[i - JANELA_BASELINE: i]
        try:
            fevd = VAR(janela).fit(LAG_VAR).fevd(HORIZONTE_FEVD)
            d = fevd.decomp[idx_fed, -1, :]
            datas.append(df.index[i - 1])
            linhas.append([d[idx_fed], d[idx_ted], d[idx_un]])
        except:
            continue
            
    df_fevd = pd.DataFrame(np.array(linhas) * 100, index=datas,
                           columns=['Choques Próprios (Fed)', 'TED Spread', 'Desemprego'])
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.stackplot(df_fevd.index, df_fevd['Choques Próprios (Fed)'], df_fevd['TED Spread'], df_fevd['Desemprego'],
                 labels=['Choques Próprios (Fed)', 'Risco de Liquidez (TED Spread)', 'Economia Real (Desemprego)'],
                 colors=['#2ca02c', '#d62728', '#1f77b4'], alpha=0.85)
                 
    ax.axvline(pd.to_datetime('2007-08-01'), color='orange', linestyle='--', linewidth=1.5, label='Início Subprime')
    ax.axvline(pd.to_datetime('2008-09-15'), color='black', linestyle=':', linewidth=2.0, label='Quebra Lehman Bros.')
    ax.set_ylabel('Proporção explicada (%)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Ano', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
    
    plt.tight_layout()
    plt.savefig('09b_painel_fevd_empilhada_corrigido.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("[SUCESSO] Gráficos da Seção 4.3 gerados e salvos com sucesso.")

# ==========================================
# 3. OUTPUTS DO APÊNDICE A.2 (Painéis Multijanela de Robustez)
# ==========================================
def gerar_painels_robustez_apendice():
    print("[*] Gerando painéis de robustez multijanela para o Apêndice A.2...")
    
    # A. Painel IRF Multijanela (36 a 84m)
    fig_irf, eixos_irf = plt.subplots(len(LISTA_JANELAS), len(REGIMES_IRF), figsize=(16, 12), sharey=True, sharex=True)
    meses = np.arange(HORIZONTE_IRF + 1)

    for i, w in enumerate(LISTA_JANELAS):
        for j, (nome, data_fim_str) in enumerate(REGIMES_IRF.items()):
            data_fim = pd.to_datetime(data_fim_str)
            data_inicio = data_fim - pd.DateOffset(months=w)
            janela = df.loc[data_inicio:data_fim]

            res = VAR(janela).fit(LAG_VAR)
            irf = res.irf(HORIZONTE_IRF)
            resposta = irf.orth_irfs[:, idx_fed, idx_ted] 
            inf, sup = irf.errband_mc(orth=True, repl=N_REPLICACOES, signif=SIGNIF, seed=SEMENTE)
            
            ax = eixos_irf[i, j]
            ax.plot(meses, resposta, color=CORES_REGIMES[nome], linewidth=2.5)
            ax.fill_between(meses, inf[:, idx_fed, idx_ted], sup[:, idx_fed, idx_ted], color=CORES_REGIMES[nome], alpha=0.15)
            ax.axhline(0, color='black', linewidth=1, linestyle='--')
            
            if i == 0:
                ax.set_title(nome, fontsize=12, fontweight='bold')
            if j == 0:
                ax.set_ylabel(f"Janela: {w}m\nResposta Fed", fontsize=10, fontweight='bold')
            if i == len(LISTA_JANELAS) - 1:
                ax.set_xlabel('Meses após o choque', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('09_teste_robustez_irf.png', dpi=300, bbox_inches='tight')
    plt.close()

    # B. Painel FEVD Multijanela (36 a 84m)
    fig_fevd, eixos_fevd = plt.subplots(len(LISTA_JANELAS), 1, figsize=(12, 14), sharex=True)
    data_subprime, data_lehman = pd.to_datetime('2007-08-01'), pd.to_datetime('2008-09-15')

    for j_idx, w in enumerate(LISTA_JANELAS):
        datas, linhas = [], []
        for i in range(w, len(df)):
            janela = df.iloc[i - w: i]
            try:
                fevd = VAR(janela).fit(LAG_VAR).fevd(HORIZONTE_FEVD)
                d = fevd.decomp[idx_fed, -1, :] 
                datas.append(df.index[i - 1])
                linhas.append([d[idx_fed], d[idx_ted], d[idx_un]])
            except:
                continue

        df_fevd = pd.DataFrame(np.array(linhas) * 100, index=datas,
                               columns=['Choques Próprios (Fed)', 'TED Spread', 'Desemprego'])
        
        ax = eixos_fevd[j_idx]
        ax.stackplot(df_fevd.index, df_fevd['Choques Próprios (Fed)'], df_fevd['TED Spread'], df_fevd['Desemprego'],
                     labels=['Choques Próprios (Fed)', 'TED Spread', 'Desemprego'],
                     colors=['#2ca02c', '#d62728', '#1f77b4'], alpha=0.85)

        ax.axvline(data_subprime, color='orange', linestyle='--', linewidth=1.5)
        ax.axvline(data_lehman, color='black', linestyle=':', linewidth=2.0)
        ax.set_ylabel('Proporção (%)', fontsize=9, fontweight='bold')
        ax.set_title(f"Decomposição da Variância - Janela de {w} meses", fontsize=10, fontweight='bold')
        ax.set_ylim(0, 100)
        
        if j_idx == 0:
            ax.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=8)

    eixos_fevd[-1].set_xlabel('Ano', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('09_teste_robustez_fevd.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("[SUCESSO] Painéis de robustez do Apêndice gerados e salvos com sucesso.")

# ==========================================
# 4. CÁLCULO DE MÉTRICAS E PELT
# ==========================================
def calcular_metricas_otimizacao():
    print("\n[*] Calculando Métricas de Otimização (RMSFE) e Overfitting...")
    resultados_metricas = {}

    for w in LISTA_JANELAS:
        bics, erros_h1, erros_h3, instaveis, total_janelas = [], [], [], 0, 0
        
        for i in range(len(df) - w - HORIZONTE_PREVISAO):
            treino, teste_h1, teste_h3 = df.iloc[i:i+w], df.iloc[i+w], df.iloc[i+w+2]
            try:
                ajuste = VAR(treino).fit(LAG_VAR)
                bics.append(ajuste.bic)
                if not ajuste.is_stable(verbose=False): instaveis += 1
                total_janelas += 1
                
                previsao = ajuste.forecast(treino.values[-LAG_VAR:], steps=HORIZONTE_PREVISAO)
                erros_h1.append(np.mean((teste_h1.values - previsao[0])**2))
                erros_h3.append(np.mean((teste_h3.values - previsao[2])**2))
            except:
                continue
                
        resultados_metricas[w] = {
            'BIC': np.mean(bics), 'RMSFE_h1': np.sqrt(np.mean(erros_h1)), 
            'Instabilidade': (instaveis / total_janelas) * 100 if total_janelas > 0 else 0
        }

    print("\n--- Resultados de Otimização e Overfitting ---")
    for w, met in resultados_metricas.items():
        print(f"Janela {w}m -> BIC: {met['BIC']:.4f} | RMSFE (h=1): {met['RMSFE_h1']:.4f} | Instabilidade: {met['Instabilidade']:.1f}%")

    datas_hip = pd.to_datetime(['2000-01-01', '2008-09-01', '2013-03-01', '2016-12-01', '2026-06-01'])
    duracoes_meses = (datas_hip[1:] - datas_hip[:-1]).days / 30.437
    media_crise = np.mean(duracoes_meses[1:3])
    vencedora = min(LISTA_JANELAS, key=lambda x: abs(x - media_crise))
    print(f"=> Duração média dos ciclos agudos da crise hipotecária: {media_crise:.1f} meses")
    print(f"=> DECISÃO ESTRUTURAL: A janela de {vencedora} meses é a ideal.")

if __name__ == "__main__":
    gerar_outputs_secao_43()
    gerar_painels_robustez_apendice()
    calcular_metricas_otimizacao()